# TRENIRANJE LONG SHORT-TERM MEMORY MODELA

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Učitaj podatke
df = pd.read_csv("output/spojeni_podaci.csv")
features = ["insolacija", "naoblaka", "temperatura", "Peak Power (kW)"]
target = "PV Yield (kWh)"
df = df[features + [target]].dropna()

# Skaliranje
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[features])
y_scaled = scaler_y.fit_transform(df[[target]]).flatten()

# Pretvori u vremenske sekvence (LSTM očekuje 3D ulaz: [batch, seq, feature])
def create_sequences(X, y, seq_length=20): #veci sequence length da vidimo kak ce ispast
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y)


In [38]:
# Podjela na train/test
split_idx = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]
# Pretvori u tenzore
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Dataset i DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)

In [39]:
# Definicija LSTM modela
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.BatchNorm1d(hidden_size),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # zadnji output u sekvenci
        out = self.fc(out)
        return out

# Inicijalizacija modela
model = LSTMModel(input_size=X.shape[1], hidden_size=128, num_layers=2)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [40]:
# Treniranje
n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_dl:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}")


Epoch 10/100, Loss: 0.4563
Epoch 20/100, Loss: 1.1352
Epoch 30/100, Loss: 0.3379
Epoch 40/100, Loss: 0.3225
Epoch 50/100, Loss: 0.1899
Epoch 60/100, Loss: 0.8340
Epoch 70/100, Loss: 0.0808
Epoch 80/100, Loss: 0.6203
Epoch 90/100, Loss: 0.1466
Epoch 100/100, Loss: 0.0664


In [ ]:
# Evaluacija
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).numpy()
    y_true_scaled = y_test_tensor.numpy()

# Inverzna transformacija
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X = scaler_X.fit_transform(df[features])
y = scaler_y.fit_transform(df[[target]]).flatten()

y_pred_inv = scaler_y.inverse_transform(y_pred_scaled)
y_true_inv = scaler_y.inverse_transform(y_true_scaled)

# Metričke evaluacije
print("\nEvaluacija LSTM modela:")
print(f"MAE: {mean_absolute_error(y_true_inv, y_pred_inv):.2f}")
print(f"MSE: {mean_squared_error(y_true_inv, y_pred_inv):.2f}")
print(f"R²: {r2_score(y_true_inv, y_pred_inv):.2f}")


Evaluacija LSTM modela:
MAE: 17.41
MSE: 538.42
R²: 0.27
